# 02 — Feature Engineering
### RetailX Demand Forecasting & Inventory Optimization Platform
**CRISP-DM Phase 4 — Feature Engineering**

Input: `data/processed/rossmann_train_store_merged.parquet`, `data/processed/rossmann_test_store_merged.parquet`
Output: modeling-ready feature tables (`data/processed/rossmann_train_features.parquet`, `rossmann_test_features.parquet`)

This notebook applies the feature groups defined in
[`docs/Phase4_Feature_Engineering_Strategy.md`](../docs/Phase4_Feature_Engineering_Strategy.md) —
calendar rhythm, promotion/holiday context, competition exposure, and lag/rolling/expanding
sales momentum — using the reusable functions in `src/feature_engineering/`. It does **not**
perform EDA, statistical testing, or model training — only feature construction and validation
of the leakage safeguards those features depend on.

## 0. Environment Setup

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR.parent) not in sys.path:
    sys.path.insert(0, str(SRC_DIR.parent))

from src.utils.logging_utils import get_logger
from src.utils.io_utils import save_parquet
from src.utils.profiling_utils import memory_usage_mb
from src.feature_engineering.calendar_features import add_calendar_features
from src.feature_engineering.promo_holiday_features import (
    add_holiday_indicators, add_promo2_active_flag, add_promo_duration,
)
from src.feature_engineering.competition_features import add_competition_features
from src.feature_engineering.lag_rolling_features import (
    add_lag_rolling_features, build_customer_traffic_lookup, apply_customer_traffic_lookup,
)
from src.feature_engineering.pipeline import build_feature_table

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
LOG_DIR = PROJECT_ROOT / "logs"

logger = get_logger("feature_engineering", LOG_DIR)

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)

logger.info("Environment ready. PROJECT_ROOT=%s", PROJECT_ROOT)

2026-07-22 11:17:51 | INFO     | feature_engineering | Environment ready. PROJECT_ROOT=C:\Users\hp\OneDrive\Desktop\Code\Retail Demand Forecasting & Inventory Optimization Platform


## 1. Load Phase 3 Output

**Objective:** Load the validated, merged Rossmann train/test datasets produced by
[`01_data_preparation.ipynb`](01_data_preparation.ipynb).

**Business context:** Feature engineering must start from data that has already passed the
Phase 3 quality gates (no nulls in `Sales`, no negative sales, verified referential integrity) —
building features on top of unvalidated data would silently propagate any upstream defect into
every downstream model.

In [2]:
train = pd.read_parquet(PROCESSED_DIR / "rossmann_train_store_merged.parquet")
test = pd.read_parquet(PROCESSED_DIR / "rossmann_test_store_merged.parquet")

print("train:", train.shape)
print("test: ", test.shape)
assert train["Sales"].isnull().sum() == 0, "Phase 3 output should never contain null Sales"
assert (train["Sales"] < 0).sum() == 0, "Phase 3 output should never contain negative Sales" 

train: (1017209, 18)
test:  (41088, 17)


## 2. Calendar Features

**Objective:** Derive weekly/monthly/quarterly/yearly rhythm features from `Date` (Section 2.1
of the feature engineering strategy doc).

**Business value:** RetailX demand is strongly periodic — weekends, month boundaries, and
seasons drive very different footfall than an ordinary midweek day. These features let a model
learn that rhythm explicitly instead of re-discovering it purely from noisy daily lags.

In [3]:
train_feat = add_calendar_features(train)
test_feat = add_calendar_features(test)

calendar_cols = ["month", "week_of_year", "quarter", "year", "day_of_month",
                  "is_weekend", "is_month_start", "is_month_end"]
train_feat[["Date", "DayOfWeek"] + calendar_cols].head()

,Date,DayOfWeek,month,week_of_year,quarter,year,day_of_month,is_weekend,is_month_start,is_month_end
0,2015-07-31,5,7,31,3,2015,31,0,0,1
1,2015-07-31,5,7,31,3,2015,31,0,0,1
2,2015-07-31,5,7,31,3,2015,31,0,0,1
3,2015-07-31,5,7,31,3,2015,31,0,0,1
4,2015-07-31,5,7,31,3,2015,31,0,0,1


**Key observations:** *(fill in after running)* — `is_weekend` should be 1 exactly on
`DayOfWeek` 6/7 (Kaggle's Sat/Sun encoding); `is_month_start`/`is_month_end` should each be 1 on
at most one row per store per month.

## 3. Promotion & Holiday Features

**Objective:** Turn the raw `Promo`/`StateHoliday`/`SchoolHoliday`/`Promo2*` columns into
model-ready indicators, including the derived `promo2_active_today` flag and `promo_duration_days`
streak counter (Section 2.2).

**Business value:** `Promo2 == 1` only means a store is *enrolled* in the recurring campaign —
`promo2_active_today` resolves whether a cycle is actually running on this specific date, which
is what actually drives demand. `promo_duration_days` separates the high-novelty first day of a
promo from a sustained run, whose uplift typically decays.

In [4]:
train_feat = add_holiday_indicators(train_feat)
train_feat = add_promo2_active_flag(train_feat)
train_feat = add_promo_duration(train_feat)

test_feat = add_holiday_indicators(test_feat)
test_feat = add_promo2_active_flag(test_feat)
test_feat = add_promo_duration(test_feat)

promo_cols = ["is_state_holiday", "state_holiday_type", "school_holiday_active",
              "promo2_active_today", "promo_duration_days"]
train_feat[["Store", "Date", "Promo", "Promo2"] + promo_cols].head(10)

,Store,Date,Promo,Promo2,is_state_holiday,state_holiday_type,school_holiday_active,promo2_active_today,promo_duration_days
0,1,2015-07-31,1,0,0,0,1,0,5
1,2,2015-07-31,1,1,0,0,1,1,5
2,3,2015-07-31,1,1,0,0,1,1,5
3,4,2015-07-31,1,0,0,0,1,0,5
4,5,2015-07-31,1,0,0,0,1,0,5
5,6,2015-07-31,1,0,0,0,1,0,5
6,7,2015-07-31,1,0,0,0,1,0,5
7,8,2015-07-31,1,0,0,0,1,0,5
8,9,2015-07-31,1,0,0,0,1,0,5
9,10,2015-07-31,1,0,0,0,1,0,5


In [5]:
print("promo2_active_today rate (train):", train_feat["promo2_active_today"].mean().round(3))
print("Stores ever showing promo2_active_today=1:", train_feat.loc[train_feat['promo2_active_today']==1, 'Store'].nunique())
print("promo_duration_days distribution (Promo==1 rows):")
print(train_feat.loc[train_feat["Promo"] == 1, "promo_duration_days"].describe())

promo2_active_today rate (train): 0.149
Stores ever showing promo2_active_today=1: 571
promo_duration_days distribution (Promo==1 rows):
count    388080.000000
mean          2.999072
std           1.414543
min           1.000000
25%           2.000000
50%           3.000000
75%           4.000000
max           5.000000
Name: promo_duration_days, dtype: float64


**Key observations:** *(fill in after running)* — `promo2_active_today` should only ever
be 1 for stores with `Promo2 == 1`; `promo_duration_days` should reset to 1 at the start of every
new promo streak and to 0 on every non-promo day.

## 4. Competition Features

**Objective:** Convert `CompetitionDistance`/`CompetitionOpenSince*` into an explicit
`has_competition` flag, a sentinel-filled distance, and a competition-duration feature
(Section 2.3).

**Business value:** A missing `CompetitionDistance` means *no competitor is tracked*, not an
unknown distance — treating it as a flag rather than imputing a fabricated number preserves that
distinction for the model. `competition_open_since_days` lets a model learn that demand erosion
from a new competitor typically grows for a period after opening, then stabilizes.

In [6]:
train_feat = add_competition_features(train_feat)
test_feat = add_competition_features(test_feat)

competition_cols = ["has_competition", "competition_distance", "competition_open_since_days"]
train_feat[["Store"] + competition_cols].drop_duplicates(subset="Store").describe()

,Store,has_competition,competition_distance,competition_open_since_days
count,1115.00000,1115.000000,1115.000000,758.000000
mean,558.00000,0.997309,5594.466309,2343.817871
std,322.01708,0.051824,8479.266602,2259.706787
min,1.00000,0.000000,20.000000,30.000000
25%,279.50000,1.000000,720.000000,972.000000
50%,558.00000,1.000000,2330.000000,2037.000000
75%,836.50000,1.000000,6905.000000,3408.000000
max,1115.00000,1.000000,75860.000000,42214.000000


**Key observations:** *(fill in after running)* — `has_competition` should be 0 for
exactly the stores with originally-missing `CompetitionDistance` (3 stores, per Phase 2/3
findings); `competition_open_since_days` should never be negative (rows before a competitor's
open date are NaN, not negative, by construction).

## 5. Lag, Rolling, and Expanding Sales-Momentum Features

**Objective:** Attach per-store historical-momentum features — `sales_lag_7/14/28`,
`sales_rolling_mean/std_7/30`, `sales_expanding_mean`, `sales_trend_7_30` — plus a training-only
`avg_customers_store_dow` traffic lookup (Section 2.4).

**Leakage safeguards enforced here** (Section 3 of the strategy doc):
- every window is computed on `shift(1)` of a closed-day-masked sales series, so a row's own
  sales value can never leak into its own rolling/expanding statistics;
- each store's series is reindexed onto a *complete* daily calendar first, so a physical gap in
  that store's raw rows can't silently shift what "7 days ago" means;
- `test` features are computed by walking each store's *training* history forward into the test
  dates — rows whose lookback window falls entirely inside the unknown test period become `NaN`
  by construction, since no forecasted value is available to fill them at this stage;
- `Customers` itself is never used as a raw per-row feature (it doesn't exist in `test.csv`) —
  only the pre-aggregated `avg_customers_store_dow` lookup, built from training data only, is
  attached to both train and test.

In [7]:
train_feat, test_feat = add_lag_rolling_features(train_feat, test_feat)

customer_lookup = build_customer_traffic_lookup(train)
train_feat = apply_customer_traffic_lookup(train_feat, customer_lookup)
test_feat = apply_customer_traffic_lookup(test_feat, customer_lookup)

momentum_cols = ["sales_lag_7", "sales_lag_14", "sales_lag_28",
                  "sales_rolling_mean_7", "sales_rolling_mean_30",
                  "sales_rolling_std_7", "sales_rolling_std_30",
                  "sales_expanding_mean", "sales_trend_7_30", "avg_customers_store_dow"]
train_feat.sort_values(["Store", "Date"])[["Store", "Date", "Sales"] + momentum_cols].tail(10)

,Store,Date,Sales,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_30,sales_rolling_std_7,sales_rolling_std_30,sales_expanding_mean,sales_trend_7_30,avg_customers_store_dow
11149,1115,2015-07-22,5342,6039.0,5900.0,5463.0,6487.333496,6707.807617,989.807617,1556.897217,6288.177246,0.967132,395.818182
10034,1115,2015-07-23,6150,6590.0,5686.0,5015.0,6371.166504,6717.269043,1088.890503,1547.428833,6286.953613,0.948476,405.040323
8919,1115,2015-07-24,5816,7874.0,5844.0,5549.0,6297.833496,6755.461426,1086.017700,1519.379272,6286.776367,0.932258,444.263566
7804,1115,2015-07-25,6897,7264.0,7164.0,6676.0,5954.833496,6769.038574,766.699524,1508.909180,6286.168945,0.879716,482.947761
6689,1115,2015-07-26,0,NaN,NaN,NaN,5893.666504,6841.422852,646.599304,1465.928589,6286.956055,0.861468,NaN
5574,1115,2015-07-27,10712,6083.0,10598.0,11006.0,5893.666504,6893.120117,646.599304,1471.769043,6286.956055,0.855007,466.140625
4459,1115,2015-07-28,8093,5074.0,7562.0,8610.0,6665.166504,7054.560059,2083.250732,1656.698853,6292.651367,0.944803,400.895522
3344,1115,2015-07-29,7661,5342.0,6039.0,7701.0,7168.333496,7094.500000,1984.313721,1635.952393,6294.965332,1.010407,395.818182
2229,1115,2015-07-30,8405,6150.0,6590.0,6858.0,7554.833496,6965.846191,1771.916382,1435.259521,6296.718750,1.084554,405.040323
1114,1115,2015-07-31,8680,5816.0,7874.0,7412.0,7930.666504,6957.961426,1649.252075,1426.401855,6299.421875,1.139797,444.263566


**Key observations:** *(fill in after running)* — nulls in the lag/rolling columns
are expected only at the start of each store's history (not enough prior days yet) and, for
`test`, only once a lookback window reaches beyond the last known training date.

## 6. Leakage Validation

**Objective:** Prove, not just assert in prose, that the momentum features respect the "no
current-day leakage" safeguard before trusting them in any model.

**Why this matters:** a rolling/lag feature that accidentally includes the current day's own
sales would make every backtest look artificially accurate and would be silently useless (or
actively misleading) once deployed against genuinely unknown future days.

In [8]:
sample_stores = train_feat["Store"].drop_duplicates().sample(25, random_state=42)

max_abs_diff = 0.0
for store_id in sample_stores:
    grp = train_feat.loc[train_feat["Store"] == store_id].sort_values("Date")

    # Independently reproduce the production logic: reindex onto a complete daily
    # calendar, mask closed-day sales as NaN, then shift(1) before rolling -- mirroring
    # src/feature_engineering/lag_rolling_features.py without importing it, so this is a
    # genuine independent check rather than the pipeline checking itself. If the pipeline
    # had leaked the current day into its own window, this exact reproduction would diverge
    # from it far more than floating-point noise.
    full_idx = pd.date_range(grp["Date"].min(), grp["Date"].max(), freq="D")
    masked = grp.set_index("Date")["Sales"].where(grp.set_index("Date")["Open"] == 1).reindex(full_idx)
    expected = masked.shift(1).rolling(7, min_periods=1).mean()

    actual = grp.set_index("Date")["sales_rolling_mean_7"].reindex(full_idx)
    diff = (actual - expected).abs()
    if diff.notna().any():
        max_abs_diff = max(max_abs_diff, diff.max())

print("Max abs difference between pipeline and independently recomputed rolling_mean_7:", max_abs_diff)
assert max_abs_diff < 1e-3, "Rolling feature does not match an independently recomputed shift(1) rolling mean"
print("Leakage check passed: sales_rolling_mean_7 never includes the current row's own Sales.")

Max abs difference between pipeline and independently recomputed rolling_mean_7: 0.0007812500007275958
Leakage check passed: sales_rolling_mean_7 never includes the current row's own Sales.


## 7. Full Pipeline Consistency Check

**Objective:** Confirm that running the packaged `build_feature_table()` pipeline end-to-end
(as later phases will call it) produces the same result as the step-by-step build above.

In [9]:
pipeline_train, pipeline_test = build_feature_table(train, test)

pd.testing.assert_frame_equal(
    train_feat.sort_values(["Store", "Date"]).reset_index(drop=True),
    pipeline_train.sort_values(["Store", "Date"]).reset_index(drop=True),
    check_like=True,
)
pd.testing.assert_frame_equal(
    test_feat.sort_values(["Store", "Date"]).reset_index(drop=True),
    pipeline_test.sort_values(["Store", "Date"]).reset_index(drop=True),
    check_like=True,
)
print("build_feature_table() output matches the step-by-step notebook build exactly.")

build_feature_table() output matches the step-by-step notebook build exactly.


## 8. Final Feature Table Summary

**Objective:** Profile the finished feature tables — shape, memory, and a null-count summary
tying every remaining null back to a documented, expected cause.

In [10]:
print("train_feat:", train_feat.shape, f"{memory_usage_mb(train_feat):.2f} MB")
print("test_feat: ", test_feat.shape, f"{memory_usage_mb(test_feat):.2f} MB")

null_summary = pd.DataFrame({
    "train_nulls": train_feat.isnull().sum(),
    "test_nulls": test_feat.reindex(columns=train_feat.columns).isnull().sum(),
})
null_summary[(null_summary["train_nulls"] > 0) | (null_summary["test_nulls"] > 0)]

train_feat: (1017209, 44) 125.24 MB
test_feat:  (41088, 43) 5.14 MB


,train_nulls,test_nulls
Sales,0,41088
Customers,0,41088
CompetitionDistance,2642,96
CompetitionOpenSinceMonth,323348,15216
CompetitionOpenSinceYear,323348,15216
Promo2SinceWeek,508031,17232
Promo2SinceYear,508031,17232
PromoInterval,508031,17232
competition_open_since_days,408014,15216
sales_lag_7,180608,35925


**Key observations:** *(fill in after running)* — every remaining null should trace
to one of: (a) structural `store.csv` missingness carried over from Phase 3
(`CompetitionDistance`/`CompetitionOpenSince*`/`Promo2Since*`/`PromoInterval`), (b) a store's
lag/rolling window not yet having enough prior history, or (c) a `test` row whose lookback window
falls inside the unknown forecast horizon. Any null outside these three categories would indicate
a bug, not an expected gap.

## Save Feature Tables

Persist both feature tables as Parquet for direct use by Phase 5 (SQL/EDA cross-checks) and
Phase 7 (Prophet/XGBoost/LightGBM training).

In [11]:
train_output_path = PROCESSED_DIR / "rossmann_train_features.parquet"
test_output_path = PROCESSED_DIR / "rossmann_test_features.parquet"

save_parquet(train_feat, train_output_path)
save_parquet(test_feat, test_output_path)

print("Saved:", train_output_path)
print("Saved:", test_output_path)

Saved: C:\Users\hp\OneDrive\Desktop\Code\Retail Demand Forecasting & Inventory Optimization Platform\data\processed\rossmann_train_features.parquet
Saved: C:\Users\hp\OneDrive\Desktop\Code\Retail Demand Forecasting & Inventory Optimization Platform\data\processed\rossmann_test_features.parquet


## Summary

| Metric | Value |
|---|---|
| Feature groups added | Calendar (8), Promotion/Holiday (5), Competition (3), Lag/Rolling/Expanding (9), Customer traffic (1) |
| Leakage safeguards | `shift(1)`-before-window, per-store gap-safe reindexing, train-only customer lookup — all independently verified in Section 6 |
| Consistency check | Step-by-step build matches `build_feature_table()` pipeline output exactly (Section 7) |
| Output | `rossmann_train_features.parquet`, `rossmann_test_features.parquet` in `data/processed/` |

**Next phase (not started here):** Phase 7 — Machine Learning (Prophet / XGBoost / LightGBM), see
[`docs/Phase7_Machine_Learning_Strategy.md`](../docs/Phase7_Machine_Learning_Strategy.md).